# Xarray-Based Pre-Processing and Dataloading

Before training a model, we need:

1. A dataset
2. Pre-processing steps to prepare the data
3. A dataloader to feed the data to the model during training

POPSIM adopts Xarray as the primary platform for doing all of this. Xarray is an open-source library developed by geoscientists for working with multi-dimensional data, and offers a power set of tools for data processing and visualization. It has become a core component of the [Pangeo platform for "Big Data Geoscience"](https://pangeo.io/), and also helps to address many of the issues we encounter working with fusion datasets.

For a proper introduction and tutorials, check out [their documentation](https://docs.xarray.dev/en/stable/#). The scope of this tutorial is to show an example pipeline to go from a data file saved on disk to a ML-ready dataset.

## Data Pre-Processing

### Loading an Alcator C-Mod dataset
A small dataset of Alcator C-Mod shots is committed to the POPSIM repo for demonstration purposes. Let's load it up.

Below, you'll see that we have a `xr.Dataset` with dimensions (`time_slice`, `shot`) and coordinates (`time_slice`, `time`, `shot`). Note that `time_slice` is just an index, and the `time` coordinate has the actual time values because each shot might have a different timebase.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import xarray as xr

import popsim

path_to_data = os.path.join(popsim.DATA_DIR, "cmod/saperstein_july_2024_subset.nc")
ds = xr.open_dataset(path_to_data)
ds

### Plotting Power Balance Quantities
For this example, we want to train a power balance module, so let's just grab the relevant subset of the dataset. This can be done quite easily with dictionary indexing, like shown below.

In addition, Xarray has very tight integration with powerful data visualization toolchains like `hvplot`. Let's generate an interactive plot where we can view the power balance quantities for each shot.

In [ ]:
import hvplot.xarray

power_balance_vars = ["Wmhd", "p_oh", "p_rad", "p_icrf", "p_lh"]
ds_pb = ds[power_balance_vars]
ds_pb.hvplot.line(x="time", groupby="shot")

### Making Subplots with Plot Addition
Okay, that's nice, but it seems perhaps stored energy doesn't belong in the same plot as the other quantities. We can put them on separate subplots by simply creating two plots and adding them together.

In [ ]:
ds_pb.hvplot.line(x="time", y=["p_oh", "p_rad", "p_icrf", "p_lh"], groupby="shot") + ds_pb.hvplot.line(x="time", y="Wmhd", groupby="shot")

### Filtering Out Bad Data 

We can see that many of the shots have a couple issues:

1. Negative stored energy
2. Unrealistically high stored energy
3. Unrealistically large values for radiated power

So it probably makes sense to filter out this bad data. To decide a threshold, let's take a look at some stats.

**Oh yeah, you can use Pandas!** One of the nice features that Pandas has is the `describe()` function for getting some quick stats. Xarray doesn't have such a function, but we can easily convert the energy variable real quick to a Pandas DataFrame and use that function. Below, we can clearly see that a lot of data points are negative, and also that 99.99% of data is below ~2e5J. We can use this information to inform our lower and upper bounds for the stored energy.

In [ ]:
print("Wmhd")
print(ds_pb["Wmhd"].stack(sample=("shot", "time_slice")).to_pandas().describe(percentiles=(0.001, 0.25, 0.75, 0.999)))
print("p_rad")
print(ds_pb["p_rad"].stack(sample=("shot", "time_slice")).to_pandas().describe(percentiles=(0.001, 0.25, 0.75, 0.999)))

#### Defining a Mask
We can easily create a new boolean variable that serves as a mask for threshold violation and visualize it along with the time series.

In [ ]:
energy_bounds = (1e4, 2.5e5)
ds_pb["energy_in_bounds"] = (energy_bounds[0] < ds_pb["Wmhd"]) & (ds_pb["Wmhd"] < energy_bounds[1])

prad_bounds = (0.0, 1.0e7)
ds_pb["prad_in_bounds"] = (prad_bounds[0] < ds_pb["p_rad"]) & (ds_pb["p_rad"] < prad_bounds[1])

# ds_pb["values_in_bounds"] = ds_pb["energy_in_bounds"] & ds_pb["prad_in_bounds"]
ds_pb["values_in_bounds"] = ds_pb["energy_in_bounds"]

# Make the plot of energy and where it is in bounds.
energy_plot = (
    ds_pb.hvplot.line(x="time", y="Wmhd", groupby="shot") * ds_pb.hvplot.line(x="time", y="energy_in_bounds", groupby="shot")
).opts(multi_y=True)

# Make the plot of prad and where it is in bounds.
prad_plot = (ds_pb.hvplot.line(x="time", y="p_rad", groupby="shot") * ds_pb.hvplot.line(x="time", y="prad_in_bounds", groupby="shot")).opts(
    multi_y=True
)

(energy_plot + prad_plot).cols(1)

#### Masking the Largest Group
We see from the previous cell that, for some shots, the mask is not contiguous due to outlier measurements. There are several ways to handle this, one of which is to simply find the largest group of contiguous data. This isn't a core Xarray feature, but a simple helper function for this has been included in this repo:

In [ ]:
from popsim.ml.preprocess_utils import mask_to_largest_group_mask

ds_pb["values_in_bounds_largest_group"] = mask_to_largest_group_mask(ds_pb["values_in_bounds"], episode_dim="shot", time_dim="time_slice")

ds_pb.hvplot.line(x="time", y=["values_in_bounds", "values_in_bounds_largest_group"], groupby="shot")

#### Dropping Data
Now that we have a mask of the data we want to keep, we can simply use the `where` method to drop all the data we don't want.

In [ ]:
# Only keep data where the mask is True.
ds_pb_after = ds_pb.where(ds_pb["values_in_bounds_largest_group"], drop=True)

# Compare the data before and after dropping.
ds_pb.hvplot.line(x="time", y="Wmhd", groupby="shot") * ds_pb_after.hvplot.line(x="time", y="Wmhd", groupby="shot")

We are happy with `ds_pb_after`, so we'll override the original dataset with the new one and take a look at our final plot.

In [ ]:
ds_pb = ds_pb_after
ds_pb.hvplot.line(x="time", y=["p_oh", "p_rad", "p_icrf", "p_lh"], groupby="shot") + ds_pb.hvplot.line(x="time", y="Wmhd", groupby="shot")

## Data Loading
Now that the dataset is ready, we can move on to dataloading. The purpose of this step is to setup the plumbing to feed data into the model during training and validation.

First, we will split our existing dataset into a training and validation set by shot.

In [ ]:
import jax

from popsim.ml import split_dataset_by_coords

ds_train, ds_val = split_dataset_by_coords(ds_pb, fracs=(0.7, 0.3), coord="shot", key=jax.random.PRNGKey(42))
print("Training shots:")
print(ds_train.shot)
print("Validation shots:")
print(ds_val.shot)

### Making Dataloaders
We're now ready to make dataloaders for training and validation.

The provided `make_dataloader` function does a couple things for you:
1) Splits episodes (shots) into segments that can overlap (segmenting an entire shot is helpful for training)
2) Makes sure that the model doesn't run into Nans when initializing state by shifting each segment so that all variables in "state_init_vars" are non-Nan at the start of the segment
3) Breaks the segments into batches

In [ ]:
from popsim.ml.dataloading import make_dataloader

train_dl = make_dataloader(
    ds=ds_train,
    time_var="time",
    episode_var="shot",
    state_init_vars=["Wmhd"],
    param_vars=["p_rad"],
    segment_length=500,
    segment_overlap=400,
    batch_size=1024,
    shuffle=True,  # Randomize the order of samples during training.
)

val_dl = make_dataloader(
    ds=ds_val,
    time_var="time",
    episode_var="shot",
    state_init_vars=["Wmhd"],
    param_vars=["p_rad"],
    segment_length=500,
    segment_overlap=400,
    batch_size=1024,
    shuffle=False,  # Keep the same order of samples during validation.
)

During training and validation, the dataloader allows you to iterate over the dataset like such:

In [ ]:
for batch in train_dl:
    print(batch)

### Visualizing Segments

To illustrate the concept of segments, let's cook up two different dataloaders: one with segments that have small overlap, and one with segments that have large overlap. We can then visualize the segments to see the difference.

In [ ]:
dl_small_overlap = make_dataloader(
    ds=ds_pb,
    time_var="time",
    episode_var="shot",
    state_init_vars=["Wmhd"],
    param_vars=["p_rad"],
    segment_length=250,
    segment_overlap=25,
    batch_size=None,  # batch_size = None means that the dataloader will return all samples in one batch.
)

dl_big_overlap = make_dataloader(
    ds=ds_pb,
    time_var="time",
    episode_var="shot",
    state_init_vars=["Wmhd"],
    param_vars=["p_rad"],
    segment_length=250,
    segment_overlap=100,
    batch_size=None,  # batch_size = None means that the dataloader will return all samples in one batch.
)

# Extract the xr.Dataset for a single shot after applying the dataloaders.
example_shot = 1120104005
ds_small_overlap = next(iter(dl_small_overlap)).sel(shot=example_shot)
ds_big_overlap = next(iter(dl_big_overlap)).sel(shot=example_shot)

In [ ]:
ds_small_overlap["Wmhd"].hvplot.line(x="time", by="input_batch", c="b", alpha=0.5) + ds_big_overlap["Wmhd"].hvplot.line(
    x="time", by="input_batch", c="b", alpha=0.5
)